# Dark Store Demand Forecast — SwiftMart Koramangala (PIN 560068, Bengaluru)

**Objective:** Build an hourly order-volume forecast for the next 7 days using ARIMA and Prophet.  
Report MAE and RMSE on a held-out test set. Model outputs feed the ML restocking trigger (EXP-001).

**Data:** Simulated 90 days of hourly order data calibrated to Bengaluru q-commerce patterns  
(peak lunch 12–14h, peak evening 18–21h, weekend uplift ~1.3×).


## 0. Setup & Imports

In [ ]:

# Install dependencies (run once)
# !pip install pandas numpy matplotlib seaborn statsmodels prophet scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta

# Statsmodels ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Prophet
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    print("Prophet not installed — skipping Prophet section.")
    PROPHET_AVAILABLE = False

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams.update({
    "figure.figsize": (14, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "monospace",
})
sns.set_palette("muted")
print("Libraries loaded OK")


## 1. Simulate 90-Day Hourly Order Data

Calibrated to Bengaluru quick-commerce patterns:
- PIN 560068 (Koramangala) — high-density, young professional demographic
- Dual peaks: lunch (12–14h) and evening (18–21h)
- Weekend uplift ≈ 1.3× weekday baseline
- Weekly seasonality (Mon lowest, Fri–Sun highest)
- Mild growth trend (+0.3 orders/day)
- Festival spikes on known dates (Ugadi, Holi)


In [ ]:

np.random.seed(42)

# ── Date range: 90 days hourly ────────────────────────────────────
START = pd.Timestamp("2026-01-01 00:00:00")
END   = pd.Timestamp("2026-04-01 23:00:00")
idx   = pd.date_range(start=START, end=END, freq="h")

n = len(idx)

# ── Hour-of-day demand profile (0-indexed) ────────────────────────
# Mimics typical q-commerce demand curve in Bengaluru
hour_profile = np.array([
    2,  1,  1,  1,  1,  2,   # 00–05: night trough
    5,  8, 10, 10, 11, 13,   # 06–11: morning ramp
   20, 22, 18, 14, 12, 13,   # 12–17: lunch peak + afternoon
   22, 28, 25, 18, 12,  6,   # 18–23: dinner peak + late night
], dtype=float)

# Normalise so sum = 1 (share of daily orders)
hour_weight = hour_profile / hour_profile.sum()

# ── Day-of-week multiplier ────────────────────────────────────────
# 0=Mon ... 6=Sun
dow_mult = np.array([0.85, 0.88, 0.92, 0.95, 1.10, 1.25, 1.20])

# ── Linear growth trend (~0.3 extra orders/day over store lifetime) ─
day_num   = (idx - START).days
trend     = 0.3 * day_num / 24   # per-hour contribution

# ── Base daily order volume: ~350 orders/day ──────────────────────
DAILY_BASE = 350

# ── Assemble signal ───────────────────────────────────────────────
orders = np.zeros(n)
for i, ts in enumerate(idx):
    h = ts.hour
    d = ts.dayofweek
    base = DAILY_BASE * hour_weight[h] * dow_mult[d] + trend[i]
    orders[i] = max(0, base)

# ── Festival spikes ───────────────────────────────────────────────
FESTIVALS = {
    "2026-01-26": 1.6,   # Republic Day
    "2026-03-14": 1.8,   # Holi
    "2026-03-30": 1.5,   # Ugadi
}
for date_str, mult in FESTIVALS.items():
    mask = idx.date == pd.Timestamp(date_str).date()
    orders[mask] *= mult

# ── Add noise (negative-binomial-ish via Poisson overdispersion) ──
noise = np.random.gamma(shape=orders + 1, scale=1.0) - 1
orders = np.maximum(0, orders + noise * 0.15).round().astype(int)

# ── Slot utilisation columns (derived) ───────────────────────────
# Planned pickers by hour (from capacity template)
def planned_pickers(hour, dow):
    if 2 <= hour <= 6:   return 2
    if 7 <= hour <= 9:   return 6
    if 12 <= hour <= 14 and dow in [5, 6]: return 12
    if 12 <= hour <= 14: return 9
    if 18 <= hour <= 21 and dow in [5, 6]: return 12
    if 18 <= hour <= 21: return 10
    return 7

pp = np.array([planned_pickers(ts.hour, ts.dayofweek) for ts in idx])

# Avg pick time = 3.5 min → throughput per picker per hour ≈ 17 orders
PICK_THROUGHPUT = 17   # orders / picker-hour
active_pickers = np.clip(np.ceil(orders / PICK_THROUGHPUT), 0, pp).astype(int)
idle_slot_min  = (pp - active_pickers) * 60
fill_rate      = np.where(pp > 0, 1 - idle_slot_min / (pp * 60), 0)

df = pd.DataFrame({
    "ds":               idx,
    "y":                orders,            # hourly order count (Prophet convention)
    "planned_pickers":  pp,
    "active_pickers":   active_pickers,
    "idle_slot_min":    idle_slot_min,
    "fill_rate":        fill_rate.round(4),
    "hour":             idx.hour,
    "dow":              idx.dayofweek,
    "is_weekend":       (idx.dayofweek >= 5).astype(int),
    "is_festival":      idx.normalize().isin(
                            [pd.Timestamp(d) for d in FESTIVALS]
                        ).astype(int),
}).set_index("ds")

print(f"Dataset: {len(df):,} hourly rows | {START.date()} → {END.date()}")
print(f"\nOrder stats:")
print(df["y"].describe().round(1))
print(f"\nFill-rate stats:")
print(df["fill_rate"].describe().round(3))


## 2. Exploratory Data Analysis

In [ ]:

fig, axes = plt.subplots(3, 1, figsize=(16, 11))

# — 2a. Full 90-day order series —
daily = df["y"].resample("D").sum()
axes[0].plot(daily.index, daily.values, lw=1.5, color="#2196F3")
axes[0].fill_between(daily.index, daily.values, alpha=0.15, color="#2196F3")
axes[0].set_title("Daily Order Volume — SwiftMart Koramangala (PIN 560068)", fontsize=12)
axes[0].set_ylabel("Orders / day")
for date_str in FESTIVALS:
    axes[0].axvline(pd.Timestamp(date_str), color="red", lw=1, ls="--", alpha=0.7)
axes[0].legend(["Orders", "Festival spike"], loc="upper left")

# — 2b. Avg orders by hour-of-day (weekday vs weekend) —
wk  = df[df["is_weekend"] == 0].groupby("hour")["y"].mean()
we  = df[df["is_weekend"] == 1].groupby("hour")["y"].mean()
axes[1].plot(wk.index, wk.values, marker="o", ms=4, label="Weekday", color="#4CAF50")
axes[1].plot(we.index, we.values, marker="s", ms=4, label="Weekend", color="#FF9800")
axes[1].set_title("Avg Orders by Hour-of-Day", fontsize=12)
axes[1].set_ylabel("Avg orders / hour")
axes[1].set_xlabel("Hour")
axes[1].legend()
axes[1].set_xticks(range(24))

# — 2c. Fill-rate heatmap (DOW × hour) —
fill_pivot = df.groupby(["dow", "hour"])["fill_rate"].mean().unstack()
DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
im = axes[2].imshow(fill_pivot.values, aspect="auto", cmap="RdYlGn",
                    vmin=0.3, vmax=1.0, origin="upper")
axes[2].set_yticks(range(7)); axes[2].set_yticklabels(DOW_LABELS)
axes[2].set_xticks(range(0, 24, 2)); axes[2].set_xticklabels(range(0, 24, 2))
axes[2].set_title("Slot Fill-Rate Heatmap (DOW × Hour)  |  Green = utilised, Red = idle", fontsize=12)
axes[2].set_xlabel("Hour of day")
plt.colorbar(im, ax=axes[2], fraction=0.02, pad=0.01, label="Fill rate")

plt.tight_layout()
plt.savefig("../docs/eda_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Fig saved → docs/eda_dashboard.png")


## 3. Stationarity Check & Train/Test Split

- **Train:** first 80 days (≈ 1,920 hourly observations)
- **Test:** last 10 days (240 h) — held out for evaluation
- **Forecast horizon:** next 7 days after test end


In [ ]:

# ── Train / Test split ────────────────────────────────────────────
TRAIN_DAYS = 80
TEST_DAYS  = 10

split_train = START + pd.Timedelta(days=TRAIN_DAYS)
split_test  = split_train + pd.Timedelta(days=TEST_DAYS)

train = df.loc[START : split_train - pd.Timedelta(hours=1), "y"]
test  = df.loc[split_train : split_test - pd.Timedelta(hours=1), "y"]
future_start = split_test

print(f"Train: {train.index[0].date()} → {train.index[-1].date()}  ({len(train):,} obs)")
print(f"Test:  {test.index[0].date()}  → {test.index[-1].date()}   ({len(test):,} obs)")
print(f"Forecast from: {future_start.date()} (+7 days)")

# ── Augmented Dickey-Fuller stationarity test ─────────────────────
adf_result = adfuller(train, autolag="AIC")
print(f"\nADF Test — p-value: {adf_result[1]:.4f}")
print("→ Series is", "stationary" if adf_result[1] < 0.05 else "NON-stationary (differencing needed)")

# ── ACF / PACF plots to guide ARIMA order selection ──────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 3))
# Use daily-aggregated for cleaner ACF (hourly is very noisy)
train_daily = train.resample("D").sum()
plot_acf(train_daily, lags=30, ax=axes[0], title="ACF — Daily Orders")
plot_pacf(train_daily, lags=30, ax=axes[1], title="PACF — Daily Orders", method="ywm")
plt.tight_layout()
plt.show()


## 4. SARIMA Forecast (Hourly)

**Model:** SARIMA(1,0,1)(1,1,1)[24]  
- Non-seasonal: AR(1) + MA(1) — captures short-run autocorrelation  
- Seasonal: 24-hour period (one full day) with seasonal differencing  
- Exogenous regressors: `is_weekend`, `is_festival`


In [ ]:

# Exogenous features for train / test / future
exog_train  = df.loc[train.index, ["is_weekend", "is_festival"]]
exog_test   = df.loc[test.index,  ["is_weekend", "is_festival"]]

# Build future exog (7 days)
future_idx   = pd.date_range(start=future_start, periods=7*24, freq="h")
exog_future  = pd.DataFrame({
    "is_weekend":  (future_idx.dayofweek >= 5).astype(int),
    "is_festival": 0,   # no festivals in forecast window
}, index=future_idx)

print("Fitting SARIMA(1,0,1)(1,1,1)[24] — this may take ~60 s …")

sarima_model = SARIMAX(
    train,
    exog=exog_train,
    order=(1, 0, 1),
    seasonal_order=(1, 1, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = sarima_model.fit(disp=False)
print(sarima_fit.summary().tables[0])


In [ ]:

# ── SARIMA: evaluate on test set ─────────────────────────────────
sarima_test_pred = sarima_fit.forecast(steps=len(test), exog=exog_test)
sarima_test_pred = np.maximum(0, sarima_test_pred)  # clip negatives

mae_sarima  = mean_absolute_error(test, sarima_test_pred)
rmse_sarima = np.sqrt(mean_squared_error(test, sarima_test_pred))
mape_sarima = np.mean(np.abs((test.values - sarima_test_pred.values)
                              / np.maximum(test.values, 1))) * 100

print("=" * 45)
print("SARIMA — Test Set Performance (10 days)")
print("=" * 45)
print(f"  MAE  : {mae_sarima:.2f}  orders/hour")
print(f"  RMSE : {rmse_sarima:.2f}  orders/hour")
print(f"  MAPE : {mape_sarima:.1f}%")

# ── SARIMA: forecast next 7 days ─────────────────────────────────
sarima_future = sarima_fit.forecast(steps=7*24, exog=exog_future)
sarima_future = np.maximum(0, sarima_future)
sarima_future_s = pd.Series(sarima_future.values, index=future_idx, name="sarima_forecast")

# ── Plot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(test.index, test.values,             color="#555", lw=1.2, label="Actual (test)")
ax.plot(test.index, sarima_test_pred,        color="#F44336", lw=1.5, ls="--", label="SARIMA fit (test)")
ax.plot(future_idx, sarima_future_s.values,  color="#E91E63", lw=2, label="SARIMA forecast (+7d)")
ax.axvline(future_start, color="navy", lw=1, ls=":")
ax.set_title("SARIMA — Test Fit & 7-Day Forecast", fontsize=12)
ax.set_ylabel("Orders / hour")
ax.legend()
plt.tight_layout()
plt.savefig("../docs/sarima_forecast.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Prophet Forecast

Prophet handles multiple seasonalities natively (daily + weekly) and takes holiday regressors directly — ideal for q-commerce with festival spikes.


In [ ]:

if PROPHET_AVAILABLE:
    # Prophet requires DataFrame with columns ds, y
    prophet_train = df.loc[train.index].reset_index().rename(
        columns={"ds": "ds", "y": "y"}
    )[["ds", "y"]]

    # Indian festival holidays
    festivals_df = pd.DataFrame({
        "holiday":   ["Republic Day", "Holi", "Ugadi"],
        "ds":        pd.to_datetime(["2026-01-26", "2026-03-14", "2026-03-30"]),
        "lower_window": 0,
        "upper_window": 1,
    })

    m = Prophet(
        seasonality_mode="multiplicative",     # demand scales with volume
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=False,
        holidays=festivals_df,
        changepoint_prior_scale=0.05,          # conservative — avoid overfitting
        seasonality_prior_scale=10.0,
    )
    m.add_regressor("is_weekend")

    prophet_train["is_weekend"] = prophet_train["ds"].dt.dayofweek.ge(5).astype(int)
    m.fit(prophet_train)

    # Build future dataframe (test + 7-day forecast)
    future_df = m.make_future_dataframe(periods=(TEST_DAYS + 7) * 24, freq="h")
    future_df["is_weekend"] = future_df["ds"].dt.dayofweek.ge(5).astype(int)

    forecast_df = m.predict(future_df)

    # Slice test predictions
    prophet_test_pred = (
        forecast_df.set_index("ds")
        .loc[test.index, "yhat"]
        .clip(lower=0)
    )

    mae_prophet  = mean_absolute_error(test, prophet_test_pred)
    rmse_prophet = np.sqrt(mean_squared_error(test, prophet_test_pred))
    mape_prophet = np.mean(np.abs((test.values - prophet_test_pred.values)
                                   / np.maximum(test.values, 1))) * 100

    print("=" * 45)
    print("Prophet — Test Set Performance (10 days)")
    print("=" * 45)
    print(f"  MAE  : {mae_prophet:.2f}  orders/hour")
    print(f"  RMSE : {rmse_prophet:.2f}  orders/hour")
    print(f"  MAPE : {mape_prophet:.1f}%")

    # Slice 7-day forecast
    prophet_future = (
        forecast_df.set_index("ds")
        .loc[future_idx, ["yhat", "yhat_lower", "yhat_upper"]]
        .clip(lower=0)
    )

    # ── Plot Prophet components ───────────────────────────────────
    fig = m.plot_components(forecast_df)
    fig.suptitle("Prophet Seasonality Components", y=1.01)
    plt.tight_layout()
    plt.savefig("../docs/prophet_components.png", dpi=150, bbox_inches="tight")
    plt.show()

else:
    print("Prophet not available — skipping.")
    mae_prophet = rmse_prophet = mape_prophet = None
    prophet_future = None


## 6. Model Comparison & 7-Day Forecast Plot

In [ ]:

# ── Model comparison table ────────────────────────────────────────
results = {
    "SARIMA(1,0,1)(1,1,1)[24]": {
        "MAE":  round(mae_sarima,  2),
        "RMSE": round(rmse_sarima, 2),
        "MAPE": f"{mape_sarima:.1f}%",
    },
}
if mae_prophet is not None:
    results["Prophet (mult. seasonality)"] = {
        "MAE":  round(mae_prophet,  2),
        "RMSE": round(rmse_prophet, 2),
        "MAPE": f"{mape_prophet:.1f}%",
    }

results_df = pd.DataFrame(results).T
results_df.index.name = "Model"
print("\n=== MODEL PERFORMANCE (10-day hold-out) ===")
print(results_df.to_string())
print("\nTarget threshold: MAE ≤ 15% of mean hourly demand")
mean_demand = test.mean()
threshold   = 0.15 * mean_demand
print(f"Mean hourly demand (test): {mean_demand:.1f} → MAE threshold = {threshold:.1f}")
for model, row in results.items():
    flag = "PASS ✓" if row["MAE"] <= threshold else "FAIL ✗"
    print(f"  {model}: MAE={row['MAE']} → {flag}")


In [ ]:

# ── Combined 7-day forecast plot ──────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=False)

# Top: hourly forecasts
ax = axes[0]
ax.plot(test.index, test.values, color="#333", lw=1.2, label="Actual (test)")
ax.plot(sarima_future_s.index, sarima_future_s.values,
        color="#F44336", lw=2, label="SARIMA +7d")

if prophet_future is not None:
    ax.plot(prophet_future.index, prophet_future["yhat"].values,
            color="#2196F3", lw=2, label="Prophet +7d")
    ax.fill_between(prophet_future.index,
                    prophet_future["yhat_lower"],
                    prophet_future["yhat_upper"],
                    alpha=0.15, color="#2196F3", label="Prophet 80% CI")

ax.axvline(future_start, color="navy", lw=1.2, ls=":", label="Forecast start")
ax.set_title("7-Day Demand Forecast — Hourly Orders (PIN 560068)", fontsize=12)
ax.set_ylabel("Orders / hour")
ax.legend(ncol=3)

# Bottom: daily aggregated comparison
ax2 = axes[1]
sarima_daily  = sarima_future_s.resample("D").sum()
actual_recent = df.loc[test.index, "y"].resample("D").sum()

ax2.bar(actual_recent.index - pd.Timedelta(hours=6), actual_recent.values,
        width=0.4, color="#555", label="Actual (test days)")
ax2.bar(sarima_daily.index, sarima_daily.values,
        width=0.4, color="#F44336", alpha=0.8, label="SARIMA daily")

if prophet_future is not None:
    prophet_daily = prophet_future["yhat"].resample("D").sum()
    ax2.bar(prophet_daily.index + pd.Timedelta(hours=6), prophet_daily.values,
            width=0.4, color="#2196F3", alpha=0.8, label="Prophet daily")

ax2.set_title("Daily Order Volume: Test vs Forecast", fontsize=12)
ax2.set_ylabel("Orders / day")
ax2.legend()

plt.tight_layout()
plt.savefig("../docs/combined_forecast.png", dpi=150, bbox_inches="tight")
plt.show()
print("Fig saved → docs/combined_forecast.png")


## 7. Translate Forecast → Picker Slot Recommendations

The core business output: given forecast orders per hour, how many pickers should be scheduled?


In [ ]:

MAX_PICKERS      = 12      # store capacity
PICK_THROUGHPUT  = 17      # orders per picker-hour
BUFFER_FACTOR    = 1.15    # 15% headroom for variability

# Use best model (Prophet if available, else SARIMA)
if prophet_future is not None:
    best_forecast = prophet_future["yhat"].clip(lower=0)
    best_label    = "Prophet"
else:
    best_forecast = sarima_future_s
    best_label    = "SARIMA"

# Required pickers (ceil with buffer, capped at store max)
recommended_pickers = np.clip(
    np.ceil(best_forecast.values * BUFFER_FACTOR / PICK_THROUGHPUT).astype(int),
    2,   # minimum staffing (safety)
    MAX_PICKERS
)

slot_plan = pd.DataFrame({
    "forecast_orders": best_forecast.values.round(1),
    "recommended_pickers": recommended_pickers,
    "predicted_fill_rate": np.where(
        recommended_pickers > 0,
        np.minimum(1.0, best_forecast.values / (recommended_pickers * PICK_THROUGHPUT)),
        0
    ).round(3),
}, index=future_idx)

slot_plan["hour"] = slot_plan.index.hour
slot_plan["dow"]  = slot_plan.index.dayofweek

print("7-Day Picker Slot Plan — Daily Summary")
print("=" * 60)
daily_plan = slot_plan.resample("D").agg({
    "forecast_orders":       "sum",
    "recommended_pickers":   "mean",
    "predicted_fill_rate":   "mean",
})
daily_plan.index = daily_plan.index.strftime("%a %Y-%m-%d")
daily_plan.columns = ["Forecast Orders", "Avg Pickers Rec.", "Avg Fill Rate"]
daily_plan["Avg Pickers Rec."] = daily_plan["Avg Pickers Rec."].round(1)
daily_plan["Avg Fill Rate"]    = (daily_plan["Avg Fill Rate"] * 100).round(1)
print(daily_plan.to_string())

print(f"\nModel used: {best_label}")
print(f"Buffer factor: {BUFFER_FACTOR}× | Throughput: {PICK_THROUGHPUT} orders/picker-hr")

# ── Plot slot plan ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].fill_between(slot_plan.index, slot_plan["forecast_orders"],
                      alpha=0.7, color="#2196F3")
axes[0].set_ylabel("Forecast orders / hr")
axes[0].set_title(f"7-Day Picker Slot Plan ({best_label} forecast)", fontsize=12)

axes[1].step(slot_plan.index, slot_plan["recommended_pickers"],
             where="post", color="#4CAF50", lw=2, label="Recommended")
axes[1].axhline(MAX_PICKERS, color="red", lw=1, ls="--", label=f"Max ({MAX_PICKERS})")
axes[1].set_ylabel("Pickers scheduled")
axes[1].set_ylim(0, MAX_PICKERS + 1)
axes[1].legend()

plt.tight_layout()
plt.savefig("../docs/slot_plan_7day.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Export Forecast for WMS / SQL Ingestion

In [ ]:

# Export as CSV — maps directly to a `demand_forecasts` table in the WMS DB
export_df = slot_plan.reset_index().rename(columns={"index": "forecast_hour"})
export_df.insert(0, "store_id", 1)
export_df.insert(1, "pin_code", "560068")
export_df.insert(2, "model_version", f"{best_label}-v1")
export_df.insert(3, "generated_at", pd.Timestamp.now().isoformat())

export_path = "../data/forecast_7day.csv"
export_df.to_csv(export_path, index=False)
print(f"Exported {len(export_df)} rows → {export_path}")
print(export_df.head(6).to_string(index=False))

# ── Summary scorecard ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("FINAL SCORECARD")
print("=" * 55)
print(f"  Store          : SwiftMart Koramangala (PIN 560068)")
print(f"  Train period   : {train.index[0].date()} → {train.index[-1].date()}")
print(f"  Test period    : {test.index[0].date()}  → {test.index[-1].date()}")
print(f"  Forecast window: {future_idx[0].date()} → {future_idx[-1].date()}")
print()
print(f"  SARIMA  MAE  = {mae_sarima:.2f}  |  RMSE = {rmse_sarima:.2f}  |  MAPE = {mape_sarima:.1f}%")
if mae_prophet:
    print(f"  Prophet MAE  = {mae_prophet:.2f}  |  RMSE = {rmse_prophet:.2f}  |  MAPE = {mape_prophet:.1f}%")
print()
print(f"  Baseline fill-rate (hist avg) : ~62%")
print(f"  Target fill-rate (PRD)        : ≥75%")
print(f"  Predicted avg fill-rate (7d)  : {slot_plan['predicted_fill_rate'].mean()*100:.1f}%")
print("=" * 55)
